# Running Ollama in Google Colab



### Introduction: Your Local AI Workspace in the Cloud

**Welcome to your local AI development environment!**
In this notebook, we are running **Ollama**, a lightweight, powerful engine that allows you to download and run large language models ( LLMs ) completely locally. Because Ollama needs to run continuously in the background to serve our AI requests, we have spun it up inside a hidden tmux session. Think of tmux as a background control room: it keeps the Ollama server running silently in the cloud virtual machine without blocking your ability to run Python code here in the notebook. With this setup, you can write Python scripts that send queries to this background service, giving you hands-on experience building, orchestrating, and interacting with private, local AI pipelines.


### References

- [Build Your Own AI Lab]( https://learning.oreilly.com/course/build-your-own/9780135439616/ )





## Setup Ollama



In [ ]:
%%capture
%%bash
pip install ollama


In [ ]:
import requests
from time import sleep
import ollama
import textwrap


### Install Ollama


In [ ]:
%%writefile /tmp/tmux.ollama.setup.sh
#!/bin/bash

# ollama service install and launch
rm -f /tmp/.done.ollama
mkdir -p /tmp/ollama-logs/
exec > >(tee /tmp/ollama-logs/ollama.setup.log) 2>&1
until which zstd ; do
  apt-get update
  apt-get install -y zstd
done

curl -fsSL https://ollama.com/install.sh | sh
touch /tmp/.done.ollama


In [ ]:
%%bash

# Start and run the setup script
tmux new -s ollama-setup -d 'bash /tmp/tmux.ollama.setup.sh'


### Launch Ollama


In [ ]:
%%writefile /tmp/tmux.ollama.launch.sh
#!/bin/bash

mkdir -p /tmp/ollama-logs/
exec > >(tee /tmp/ollama-logs/ollama.launch.log) 2>&1

# wait for setup to finish
until [ -f /tmp/.done.ollama ] ; do
  date
  sleep 1
done

# OLLAMA_CUDA=1
OLLAMA_KEEP_ALIVE=20m ollama serve
echo == Done
sleep 10



In [ ]:
%%bash

# Start launch session
tmux new-session -s ollama-launch -d

# Run the launch script
tmux send-keys -t ollama-launch 'bash /tmp/tmux.ollama.launch.sh' Enter


### Get a model


In [ ]:
%%writefile /tmp/tmux.ollama.model.sh
#!/bin/bash

mkdir -p /tmp/ollama-logs/
exec > >(tee /tmp/ollama-logs/ollama.model.log) 2>&1

# wait for ollama service to start
until curl -s -I http://localhost:11434/api/tags ; do
  date
  sleep 1
done

# pull a small model
ollama pull llama3.2
ollama run llama3.2 ''


In [ ]:
%%bash

# Start and run the model script
tmux new-session -s ollama-model -d 'bash /tmp/tmux.ollama.model.sh'


In [ ]:
url = "http://localhost:11434/api/generate"

print("Waiting for Ollama to start")
for i in range(1,301):
  try:
    requests.head( url )
    print()
    break
  except:
    print("=", end="")
    if i % 30 == 0 :
      print(f" -- {i:3}s")
    sleep(1)
print(f"{i} seconds")

print("Open Ollama has started")


## Use via requests module


In [ ]:
url = "http://localhost:11434/api/generate"
payload = {
    "model": "llama3.2",
    "prompt": "In one sentence, what is CS50?",
    "stream": False  # Set to True if you want to iterate over a streaming response chunks
}

response = requests.post(url, json=payload)
data = response.json()

wrapped_text = textwrap.fill(data["response"], width=80)
print(wrapped_text)


## Use via ollama module


In [ ]:
response = ollama.generate(
    model='llama3.2',
    prompt='In one sentence, what is CS50?'
)

wrapped_text = textwrap.fill(data["response"], width=80)
print(wrapped_text)


## Use via the terminal

Command line argument

```bash
ollama list
```

```bash
ollama run llama3.2 "In one sentence, what is CS50?"
```

Via pipe
```bash
echo "In one sentence, what is CS50?" |
  ollama run llama3.2
```

Interactively
```bash
echo "In one sentence, what is CS50?"
ollama run llama3.2
```


